# 同步与异步请求处理

学习目标：能根据调用的 I/O 接口选择路由写法，并通过执行线程与请求顺序辨认阻塞事件循环的代码。

前置知识：线程与协程的基本区别、async/await、事件循环、FastAPI 路由与依赖注入。

运行环境：Python 3.12、FastAPI、AnyIO 4。

工作目录：`content/Web与应用开发/FastAPI`。代码从上到下执行，Notebook 直接使用 `await`。

环境准备：[安装与运行说明](README.md)。

## 1 路由函数在哪个线程执行

同步 I/O 调用会让调用它的线程等待，例如读取文件或等待同步数据库驱动返回。FastAPI 调用普通 def 路由时，会把它交给线程池执行，再等待结果；async def 路由在事件循环中执行。

先用 threading.get_ident() 返回当前线程的标识。标识的具体数值没有业务含义，这里只比较是否为同一个线程。

HTTPX 的 ASGITransport 直接调用应用。下面的请求与 Notebook 共用事件循环，无需监听端口；这能观察应用调度，不能代表真实网络吞吐量。

In [1]:
from threading import get_ident

from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient

app = FastAPI()
loop_thread = get_ident()


@app.get("/sync-thread")
def sync_thread():
    return {"thread": get_ident()}


transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url="http://lesson") as client:
    result = (await client.get("/sync-thread")).json()
print("同步路由在事件循环线程：", result["thread"] == loop_thread)  # 预期：同步路由在事件循环线程： False。
assert result["thread"] != loop_thread  # 普通路由由框架交给工作线程。

同步路由在事件循环线程： False


在同一个应用中增加 async def 路由。返回字典这类很短的操作可以直接完成，不必为了使用 async def 人为添加 await。

In [2]:
@app.get("/async-thread")
async def async_thread():
    return {"thread": get_ident()}


transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url="http://lesson") as client:
    result = (await client.get("/async-thread")).json()
print("异步路由在事件循环线程：", result["thread"] == loop_thread)  # 预期：异步路由在事件循环线程： True。
assert result["thread"] == loop_thread

异步路由在事件循环线程： True


## 2 普通辅助函数不会自动进入线程池

FastAPI 负责调度自己调用的路由与依赖；你的代码直接调用普通函数时，函数仍在调用者线程执行。helper 写成 def，并不会使阻塞操作自动进入线程池。

![谁调用函数，决定是否由 FastAPI 调度](image/illustration/06-01-call-dispatch.svg)

图示：框架调度与普通函数调用的区别；图中 helper 不经过 FastAPI 的路由或依赖调度。

下面由异步路由直接调用 helper，比较其线程编号与 loop\_thread，应为同一线程；再与前面由框架调度的 def 路由对照。

In [3]:
def helper():
    return get_ident()


@app.get("/helper-thread")
async def helper_thread():
    return {"thread": helper()}


transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url="http://lesson") as client:
    result = (await client.get("/helper-thread")).json()
print("直接调用的 helper 在事件循环线程：", result["thread"] == loop_thread)  # 预期：直接调用的 helper 在事件循环线程： True。
assert result["thread"] == loop_thread

直接调用的 helper 在事件循环线程： True


如果异步路由确实需要调用同步 I/O 函数，可以显式使用 AnyIO 的 to_thread.run_sync，把调用交给工作线程并等待返回值。下例仍用线程标识观察调用位置；实际使用时传入需要执行的同步函数及参数。

In [4]:
from anyio import to_thread


@app.get("/offloaded-helper")
async def offloaded_helper():
    return {"thread": await to_thread.run_sync(helper)}


transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url="http://lesson") as client:
    result = (await client.get("/offloaded-helper")).json()
print("显式转交的 helper 在事件循环线程：", result["thread"] == loop_thread)  # 预期：显式转交的 helper 在事件循环线程： False。
assert result["thread"] != loop_thread

显式转交的 helper 在事件循环线程： False


## 3 依赖也按声明方式调度

通过 Depends 声明的普通 def 依赖在工作线程执行，async def 依赖在事件循环执行；路由与依赖可以混用两种写法。不能据此要求多个同步依赖或同步路由始终使用同一个工作线程。

先分别定义返回线程标识的两种依赖。

In [5]:
from typing import Annotated

from fastapi import Depends


def sync_dependency():
    return get_ident()


async def async_dependency():
    return get_ident()

把两种依赖注入同一条异步路由。这里比较每个返回值与事件循环线程是否相同，不比较两次请求的工作线程编号。

In [6]:
@app.get("/dependency-threads")
async def dependency_threads(
    sync_id: Annotated[int, Depends(sync_dependency)],
    async_id: Annotated[int, Depends(async_dependency)],
):
    return {"route": get_ident(), "sync": sync_id, "async": async_id}


transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url="http://lesson") as client:
    result = (await client.get("/dependency-threads")).json()
on_loop = {name: thread == loop_thread for name, thread in result.items()}
print("是否在事件循环线程：", on_loop)  # 预期：是否在事件循环线程： {'route': True, 'sync': False, 'async': True}。
assert on_loop == {"route": True, "sync": False, "async": True}

是否在事件循环线程： {'route': True, 'sync': False, 'async': True}


## 4 相同等待任务下，其他请求能否推进

用固定的 0.08 秒等待模拟 I/O 等待阶段，每种写法接收 3 个请求。sleep 不是实际的文件或网络 I/O；这里只观察等待时的调度差异。

time.sleep 会暂停当前线程。放在普通 def 路由中时，等待发生在工作线程。用线程安全的 SimpleQueue 记录各请求的开始和结束，request_id 是本次实验的请求编号。

In [7]:
import asyncio
import time
from queue import SimpleQueue

WAIT_SECONDS = 0.08
REQUEST_COUNT = 3
events = SimpleQueue()


@app.get("/sync-wait/{request_id}")
def sync_wait(request_id: int):
    events.put((request_id, "开始"))
    time.sleep(WAIT_SECONDS)
    events.put((request_id, "结束"))
    return {"request_id": request_id}

asyncio.sleep 暂停当前协程，让事件循环运行其他任务。真实异步 I/O 库通过可等待的接口交还执行机会；调用时需要 await，不能只调用后丢掉返回的协程。

In [8]:
@app.get("/async-wait/{request_id}")
async def async_wait(request_id: int):
    events.put((request_id, "开始"))
    await asyncio.sleep(WAIT_SECONDS)
    events.put((request_id, "结束"))
    return {"request_id": request_id}

下面是阻塞反例：只把函数声明改成 async def，仍直接调用 time.sleep。等待会占住事件循环线程，同一循环中的其他请求无法在这段等待中继续执行。

反例的等待时长和请求数量保持有限，阅读时无需扩大参数。

In [9]:
@app.get("/blocking-wait/{request_id}")
async def blocking_wait(request_id: int):
    events.put((request_id, "开始"))
    time.sleep(WAIT_SECONDS)  # 故意阻塞，用来观察错误的异步写法。
    events.put((request_id, "结束"))
    return {"request_id": request_id}

用同一个请求函数发起每组 3 个请求。TaskGroup 把请求安排为并发任务，退出时等待组内任务结束；AsyncClient 的上下文负责关闭客户端。

每个请求产生两条事件，所有请求结束后再读取记录。组与组之间顺序运行，避免混入上一组事件；perf_counter 的差值以秒计，仅记录本次耗时。

In [10]:
async def observe(path):
    transport = ASGITransport(app=app)
    started = time.perf_counter()
    async with AsyncClient(transport=transport, base_url="http://lesson") as client:
        async with asyncio.TaskGroup() as group:
            tasks = [
                group.create_task(client.get(f"{path}/{request_id}"))
                for request_id in range(REQUEST_COUNT)
            ]
    elapsed = time.perf_counter() - started
    for request_id, task in enumerate(tasks):
        response = task.result()
        assert response.status_code == 200
        assert response.json() == {"request_id": request_id}
    order = [events.get_nowait() for _ in range(REQUEST_COUNT * 2)]
    assert events.empty()  # 此时本组任务已全部结束，不会再写入记录。
    return order, elapsed

顺序记录里，如果第一个“结束”之前已经出现多个“开始”，说明一个请求尚在等待时，其他请求已经进入路由。实际完成次序可能随调度改变；耗时包括框架和调度开销，不作为固定速度承诺。

In [11]:
for path in ["/sync-wait", "/async-wait", "/blocking-wait"]:
    order, elapsed = await observe(path)
    stages = [stage for _, stage in order]
    first_end = stages.index("结束")
    print(path, order)  # 预期：同步线程池与异步等待可先出现多个开始事件；阻塞写法逐个开始、结束，具体完成顺序会变化。
    print(f"首个结束前的开始数：{first_end}，本次耗时：{elapsed:.3f} 秒")  # 预期：本次三种路由的开始数依次为 3、3、1；耗时为本机测量值，不预设固定秒数或加速比。
    # 三组都检查同样的响应内容；根据事件顺序观察重叠，不断言耗时倍数。
    if path == "/blocking-wait":
        assert stages == ["开始", "结束"] * REQUEST_COUNT

/sync-wait [(0, '开始'), (1, '开始'), (2, '开始'), (2, '结束'), (0, '结束'), (1, '结束')]
首个结束前的开始数：3，本次耗时：0.089 秒
/async-wait [(0, '开始'), (1, '开始'), (2, '开始'), (0, '结束'), (1, '结束'), (2, '结束')]
首个结束前的开始数：3，本次耗时：0.108 秒


/blocking-wait [(0, '开始'), (0, '结束'), (1, '开始'), (1, '结束'), (2, '开始'), (2, '结束')]
首个结束前的开始数：1，本次耗时：0.246 秒


## 5 线程容量和 CPU 密集工作的边界

Starlette 通过 AnyIO 执行同步路由。默认线程容量为 40 个令牌；同步依赖、显式的 to_thread.run_sync 等也会使用该默认容量。令牌表示可同时使用的工作线程额度，不是提前创建了 40 个线程，也不是应用的总请求数限制。

为观察排队，把当前内核的默认额度暂时改为 1，再执行同样的 3 个同步请求。finally 恢复原值，防止这个局部实验影响后续运行。增加额度会影响内存和调度开销，应结合真实负载决定。

In [12]:
limiter = to_thread.current_default_thread_limiter()
original_tokens = limiter.total_tokens
print("原工作线程额度：", original_tokens)  # 预期：当前工作线程额度；未另行配置时本环境为 40。
try:
    limiter.total_tokens = 1
    order, elapsed = await observe("/sync-wait")
    print("额度为 1：", order)  # 预期：额度为 1 时，每个请求先开始再结束，下一个请求随后运行。
    print(f"本次耗时：{elapsed:.3f} 秒")  # 预期：一个正的实测秒数，受调度与机器负载影响。
    assert [stage for _, stage in order] == ["开始", "结束"] * REQUEST_COUNT
finally:
    limiter.total_tokens = original_tokens
print("已恢复额度：", limiter.total_tokens)  # 预期：恢复为上面记录的 original_tokens，本环境为 40。
assert limiter.total_tokens == original_tokens

原工作线程额度： 40


额度为 1： [(0, '开始'), (0, '结束'), (1, '开始'), (1, '结束'), (2, '开始'), (2, '结束')]
本次耗时：0.248 秒
已恢复额度： 40


选择写法时，先看主要工作在等待什么。CPU 密集工作是持续计算，例如大量纯 Python 数值运算；把它声明成 async def 不会让计算自动交还执行机会。

本章的 CPython 3.12 环境受 GIL（全局解释器锁）限制，同一解释器中的多个线程不能同时执行 Python 代码。长时间的纯 Python 计算若需使用多个 CPU 核心，应考虑进程或独立计算服务；部分原生扩展会释放 GIL，须按具体库判断。

| 场景 | 路由中的处理方式 | 注意点 |
| --- | --- | --- |
| 同步数据库、文件等 I/O 接口 | 普通 def 路由，或显式转交工作线程 | 会占用线程容量 |
| 支持 await 的异步 I/O 接口 | async def 路由中 await 调用 | 路径中仍不能直接混入长时间同步阻塞 |
| 简短、无需等待的操作 | 可直接使用 async def | 无需人为添加等待 |
| 长时间 CPU 密集工作 | 按计算库特性设计进程或计算服务 | 增加线程额度不能解决纯 Python 的多核计算问题 |

## 本章小结

（1）FastAPI 调用 def 路由和同步依赖时使用工作线程，async def 路由和异步依赖在事件循环中执行。

（2）普通 helper 的直接调用不会自动切换线程。异步声明本身也不会消除内部的阻塞调用。

（3）比较并发行为要固定工作量并观察事件顺序。线程容量有限；等待型 I/O 和持续计算需要分别判断。

自查：能否仅根据调用方式判断代码运行在线程池还是事件循环，并用请求事件顺序解释阻塞现象？

## 练习

1. 增加一条普通 `def` 路由，在其中直接调用 `helper`，返回路由与 helper 的线程标识。核对两者相同，并且都与事件循环线程不同。线程标识只用于同一次运行中的比较，不要写死数值。

In [ ]:
# 在此定义同步路由并请求，比较本次运行中的线程标识。

2. 新增 `/fixed-wait/{request_id}`，保留异步路由与两次事件记录，用 `await asyncio.sleep` 修复阻塞反例。用 `observe` 执行 3 个请求，检查响应编号完整，并从实际顺序中指出哪些请求的处理区间发生重叠；不设置耗时倍数要求。保留新路径，便于继续对照错误写法。

In [ ]:
# 在此增加修复后的路由，并观察三个请求的事件顺序。

3. 把默认线程额度临时设为 2，对 `/sync-wait` 运行同一组请求。按事件顺序计算“已开始但尚未结束”的数量，检查最大值不超过 2，结束后回到 0，并在 `finally` 中恢复原额度。

In [ ]:
# 在此临时调整线程额度，统计活动请求数，并恢复原值。

### 第 3 题提示与解析

提示 1：先保存 `total_tokens`，用 `try/finally` 包住临时修改和观察，确保后续实验仍使用原额度。

提示 2：依次读事件，开始时活动数加 1，结束时减 1；每步更新最大值。不要把请求编号次序写成断言。

解析：额度为 2 时，这组同步路由最多有 2 个等待尚未完成；第 3 个请求需要等待工作线程额度释放。实际顺序可变，例如开始 0、开始 1、结束 0、开始 2、结束 1、结束 2，对应活动数 1、2、1、2、1、0。检查最大值不超过 2、最终为 0，并核对三个响应编号完整；耗时只作本次观察，不设固定倍数。

## 参考与引用来源

1. **FastAPI 官方文档**：[Concurrency and async / await](https://fastapi.tiangolo.com/async/#very-technical-details) 的 Path operation functions、Dependencies、Sub-dependencies、Other utility functions；同页 In a hurry? 与 Concurrency + Parallelism 小节用于调用方式和 I/O、CPU 工作的选择。

2. **Starlette 官方文档**：[Thread Pool](https://starlette.dev/threadpool/#concurrency-limitations)，同步调用的 AnyIO 实现、共享的默认 40 个线程令牌及增加容量的影响。

3. **AnyIO 4 官方文档**：[Working with threads](https://anyio.readthedocs.io/en/stable/threads.html#running-a-function-in-a-worker-thread) 的 Running a function in a worker thread 与 Adjusting the default maximum worker thread count，显式执行同步函数、读取及调整默认容量；该容量不控制 asyncio 自己的默认线程执行器。

4. **HTTPX 官方文档**：[ASGI Transport](https://www.python-httpx.org/advanced/transports/#asgi-transport)，AsyncClient 直接请求 ASGI 应用的写法和上下文用法。

5. **Python 3.12 官方文档**：[threading](https://docs.python.org/3.12/library/threading.html#threading.get_ident) 的 get_ident 与开篇 GIL 说明；[asyncio 的 Task Groups 与 Sleeping](https://docs.python.org/3.12/library/asyncio-task.html#task-groups)；[time.sleep](https://docs.python.org/3.12/library/time.html#time.sleep) 与同页 perf_counter；[queue.SimpleQueue](https://docs.python.org/3.12/library/queue.html#queue.SimpleQueue)，线程安全的事件队列、put 与 get_nowait。